In [ ]:
from mpi4py import MPI
import gmsh
import math
from pathlib import Path

# For DolfinX import
from dolfinx.io import gmsh as gmshio
from dolfinx.io import XDMFFile

# ------------------------------------------------------------------
# 1. Generate geometry in Gmsh (square with attached semicircular bubble)
# ------------------------------------------------------------------
gmsh.initialize()
gmsh.model.add("square_with_attached_bubble")

# -------------------------
# Parameters
# -------------------------
L = 1.0
V = 0.02
r = math.sqrt(2 * V / math.pi)
lc = 0.02  # mesh size

cx = L / 2
cy = 0.0

# -------------------------
# Points
# -------------------------
p1 = gmsh.model.geo.addPoint(0, 0, 0, lc)  # bottom-left
p2 = gmsh.model.geo.addPoint(L, 0, 0, lc)  # bottom-right
p3 = gmsh.model.geo.addPoint(L, L, 0, lc)  # top-right
p4 = gmsh.model.geo.addPoint(0, L, 0, lc)  # top-left

pc = gmsh.model.geo.addPoint(cx, cy, 0, lc)      # bubble center
pl = gmsh.model.geo.addPoint(cx - r, cy, 0, lc)  # left bubble edge
pr = gmsh.model.geo.addPoint(cx + r, cy, 0, lc)  # right bubble edge
pt = gmsh.model.geo.addPoint(cx, r, 0, lc)       # top of semicircle

# -------------------------
# Lines / arcs
# -------------------------
l_bottom_left = gmsh.model.geo.addLine(p1, pl)
arc_left = gmsh.model.geo.addCircleArc(pl, pc, pt)
arc_right = gmsh.model.geo.addCircleArc(pt, pc, pr)
l_bottom_right = gmsh.model.geo.addLine(pr, p2)
l_right = gmsh.model.geo.addLine(p2, p3)
l_top = gmsh.model.geo.addLine(p3, p4)
l_left = gmsh.model.geo.addLine(p4, p1)

# -------------------------
# Curve loop & surface
# -------------------------
loop = gmsh.model.geo.addCurveLoop([
    l_bottom_left, arc_left, arc_right, l_bottom_right, l_right, l_top, l_left
])
surface = gmsh.model.geo.addPlaneSurface([loop])
gmsh.model.geo.synchronize()

# -------------------------
# Physical groups
# -------------------------
fluid_tag = gmsh.model.addPhysicalGroup(2, [surface])
bubble_tag = gmsh.model.addPhysicalGroup(1, [arc_left, arc_right])
bottom_tag = gmsh.model.addPhysicalGroup(1, [l_bottom_left, l_bottom_right])
top_tag    = gmsh.model.addPhysicalGroup(1, [l_top])
left_tag   = gmsh.model.addPhysicalGroup(1, [l_left])
right_tag  = gmsh.model.addPhysicalGroup(1, [l_right])

# Assign names
gmsh.model.setPhysicalName(2, fluid_tag, "fluid")
gmsh.model.setPhysicalName(1, bubble_tag, "bubble_boundary")
gmsh.model.setPhysicalName(1, bottom_tag, "bottom")
gmsh.model.setPhysicalName(1, top_tag, "top")
gmsh.model.setPhysicalName(1, left_tag, "left")
gmsh.model.setPhysicalName(1, right_tag, "right")

# -------------------------
# Mesh
# -------------------------
gmsh.option.setNumber("Mesh.MshFileVersion", 4.1)
gmsh.model.mesh.generate(2)

out_dir = Path("mesh_output")
out_dir.mkdir(exist_ok=True)
gmsh.write(str(out_dir / "square_bubble_attached.msh"))

# ------------------------------------------------------------------
# 2. Import into DolfinX BEFORE finalize()
# ------------------------------------------------------------------
mesh_data = gmshio.model_to_mesh(
    gmsh.model(),
    MPI.COMM_WORLD,
    rank=0,
    gdim=2
)

dolfinx_mesh = mesh_data.mesh
cell_tags = mesh_data.cell_tags
facet_tags = mesh_data.facet_tags

# ------------------------------------------------------------------
# 3. Export to XDMF
# ------------------------------------------------------------------
with XDMFFile(dolfinx_mesh.comm, out_dir / "square_bubble_mesh.xdmf", "w") as xdmf:
    xdmf.write_mesh(dolfinx_mesh)
    xdmf.write_meshtags(cell_tags, dolfinx_mesh.geometry)
    xdmf.write_meshtags(facet_tags, dolfinx_mesh.geometry)

gmsh.finalize()
print("DolfinX mesh and tags written to:", out_dir)

In [ ]:
# # Convert Gmsh mesh to DolfinX
from dolfinx.io import gmsh as gmshio
mesh_data = gmshio.read_from_msh(
    "square_bubble.msh", MPI.COMM_WORLD, 0, gdim=2
)
mesh = mesh_data.mesh
assert mesh_data.facet_tags is not None
facet_tags = mesh_data.facet_tags
facet_tags.name = "Facet markers"

assert mesh_data.cell_tags is not None
cell_tags = mesh_data.cell_tags
cell_tags.name = "Cell markers"

Info    : Reading 'square_bubble.msh'...
Info    : 16 entities
Info    : 3012 nodes
Info    : 6022 elements
Info    : Done reading 'square_bubble.msh'


In [38]:
import dolfinx
import dolfinx.fem as fem
import dolfinx.mesh
import ufl
from ufl import inner
import numpy as np
from mpi4py import MPI
from basix.ufl import element, mixed_element
from dolfinx.io import gmsh as gmshio
from dolfinx.fem.petsc import NonlinearProblem
from dolfinx.nls.petsc import NewtonSolver
from dolfinx.io import VTXWriter

# ─────────────────────────────────────────────
# 1. Load mesh
# ─────────────────────────────────────────────
mesh_data = gmshio.read_from_msh("mesh_output/square_bubble_attached.msh", MPI.COMM_WORLD, 0, gdim=2)
mesh      = mesh_data.mesh
facet_tags = mesh_data.facet_tags

x0   = mesh.geometry.x.copy()
tdim = mesh.topology.dim
fdim = tdim - 1
mesh.topology.create_connectivity(fdim, tdim)

# Bubble center (must match gmsh script)
cx, cy    = 0.5, 0.0
V0_bubble = 0.02
r0        = np.sqrt(2 * V0_bubble / np.pi)

# ─────────────────────────────────────────────
# 2. Mixed function space (ux, uy)
# ─────────────────────────────────────────────
Vx_el = element("Lagrange", mesh.basix_cell(), 1)
V_el  = mixed_element([Vx_el, Vx_el])
V     = fem.functionspace(mesh, V_el)
u     = fem.Function(V)

(ux, uy) = ufl.split(u)
(vx, vy) = ufl.TestFunctions(V)

# ─────────────────────────────────────────────
# 3. Collapsed subspaces + DOF maps
# ─────────────────────────────────────────────
V0, dof_map_0 = V.sub(0).collapse()   # dof_map_0: collapsed → parent indices
V1, dof_map_1 = V.sub(1).collapse()

# Collapsed Functions for writing to VTX
ux_func = fem.Function(V0, name="ux")
uy_func = fem.Function(V1, name="uy")

# BCs — zero ux on left/right
zero_ux = fem.Function(V0)
left_dofs_x  = fem.locate_dofs_topological((V.sub(0), V0), fdim, facet_tags.find(5))
right_dofs_x = fem.locate_dofs_topological((V.sub(0), V0), fdim, facet_tags.find(6))
bc_ux_left   = fem.dirichletbc(zero_ux, left_dofs_x,  V.sub(0))
bc_ux_right  = fem.dirichletbc(zero_ux, right_dofs_x, V.sub(0))

# BC — zero uy on bottom
zero_uy = fem.Function(V1)
bottom_dofs_y = fem.locate_dofs_topological((V.sub(1), V1), fdim, facet_tags.find(3))
bc_uy_bottom  = fem.dirichletbc(zero_uy, bottom_dofs_y, V.sub(1))

# BC — prescribed uy on top (Constant, updated each step)
delta_height = fem.Constant(mesh, dolfinx.default_scalar_type(0.0))
top_dofs_y   = fem.locate_dofs_topological(V.sub(1), fdim, facet_tags.find(4))
bc_uy_top    = fem.dirichletbc(delta_height, top_dofs_y, V.sub(1))

bubble_dofs_x = fem.locate_dofs_topological((V.sub(0), V0), fdim, facet_tags.find(2))
bubble_dofs_y = fem.locate_dofs_topological((V.sub(1), V1), fdim, facet_tags.find(2))

arc_disp_x = fem.Function(V0)   # will be updated each step
arc_disp_y = fem.Function(V1)

bc_arc_x = fem.dirichletbc(arc_disp_x, bubble_dofs_x, V.sub(0))
bc_arc_y = fem.dirichletbc(arc_disp_y, bubble_dofs_y, V.sub(1))

# Precompute the reference arc node positions once (mesh hasn't moved yet)
collapsed_idx_x = bubble_dofs_x[1]   # collapsed-space indices for ux arc DOFs
collapsed_idx_y = bubble_dofs_y[1]   # collapsed-space indices for uy arc DOFs

coords_V0 = V0.tabulate_dof_coordinates()   # shape (n_dofs_V0, 3)
coords_V1 = V1.tabulate_dof_coordinates()

# Unit radial vectors at each arc DOF (computed once from reference geometry)
arc_x_ref_x = coords_V0[collapsed_idx_x, 0] - cx   # x - cx
arc_x_ref_y = coords_V0[collapsed_idx_x, 1] - cy   # y - cy  (for ux dofs)
arc_y_ref_x = coords_V1[collapsed_idx_y, 0] - cx
arc_y_ref_y = coords_V1[collapsed_idx_y, 1] - cy   # (for uy dofs)

# Sanity check: all arc nodes should be at radius r0
assert np.allclose(np.sqrt(arc_x_ref_x**2 + arc_x_ref_y**2), r0, rtol=1e-3), \
    "Arc nodes not at expected radius — check bubble_tag_id or cx/cy/r0"

def update_arc_bcs(r_new):
    """Prescribe radial displacement dr at every arc node."""
    dr = r_new - r0
    # ux component of radial displacement: dr * (x - cx) / r0
    arc_disp_x.x.array[collapsed_idx_x] = dr * arc_x_ref_x / r0
    # uy component of radial displacement: dr * (y - cy) / r0
    arc_disp_y.x.array[collapsed_idx_y] = dr * arc_y_ref_y / r0

bcs = [bc_arc_x, bc_arc_y, bc_ux_left, bc_ux_right, bc_uy_bottom, bc_uy_top]

# ─────────────────────────────────────────────
# 4. Laplace smoothing formulation + solver
# ─────────────────────────────────────────────
F_form = (
      inner(ufl.grad(ux), ufl.grad(vx)) * ufl.dx
    + inner(ufl.grad(uy), ufl.grad(vy)) * ufl.dx
)

problem = NonlinearProblem(F_form, u, bcs=bcs,
    petsc_options_prefix="ale_",
    petsc_options={"ksp_type": "preonly", "pc_type": "lu",
                   "ksp_error_if_not_converged": True},
)

# ─────────────────────────────────────────────
# 5. VTX writer
# ─────────────────────────────────────────────
writer = VTXWriter(MPI.COMM_WORLD, "results/moving_bubble_mesh_pde.bp",
                   [ux_func, uy_func], engine="BP5")

# ─────────────────────────────────────────────
# 6. Time loop
# ─────────────────────────────────────────────
amplitude = 1
period = 5
times     = np.linspace(0, 10, 100)

for t in times:
    V_bubble_t = V0_bubble * (1 + amplitude * np.sin(2 * np.pi * t / period))
    r_new      = np.sqrt(2 * V_bubble_t / np.pi)
    dh         = r_new - r0

    # Update both arc and top BCs before solving
    update_arc_bcs(r_new)
    delta_height.value = dolfinx.default_scalar_type(dh)

    problem.solve()

    ux_vals = u.x.array[dof_map_0]
    uy_vals = u.x.array[dof_map_1]

    ux_func.x.array[:] = ux_vals
    uy_func.x.array[:] = uy_vals

    disp3 = np.column_stack([ux_vals, uy_vals, np.zeros(len(ux_vals))])
    mesh.geometry.x[:] = x0 + disp3

    writer.write(t)
    print(f"t={t:.2f}  r={r_new:.5f}  dh={dh:.6f}  "
          f"max|ux|={np.abs(ux_vals).max():.5f}  max|uy|={np.abs(uy_vals).max():.5f}")

writer.close()

Info    : Reading 'mesh_output/square_bubble_attached.msh'...
Info    : 16 entities
Info    : 3012 nodes
Info    : 6022 elements
Info    : Done reading 'mesh_output/square_bubble_attached.msh'
t=0.00  r=0.11284  dh=0.000000  max|ux|=0.00000  max|uy|=0.00000
t=0.10  r=0.11977  dh=0.006929  max|ux|=0.00693  max|uy|=0.00693
t=0.20  r=0.12621  dh=0.013377  max|ux|=0.01338  max|uy|=0.01338
t=0.30  r=0.13215  dh=0.019316  max|ux|=0.01932  max|uy|=0.01932
t=0.40  r=0.13756  dh=0.024722  max|ux|=0.02472  max|uy|=0.02472
t=0.51  r=0.14241  dh=0.029575  max|ux|=0.02958  max|uy|=0.02958
t=0.61  r=0.14669  dh=0.033855  max|ux|=0.03385  max|uy|=0.03385
t=0.71  r=0.15038  dh=0.037544  max|ux|=0.03754  max|uy|=0.03754
t=0.81  r=0.15346  dh=0.040627  max|ux|=0.04063  max|uy|=0.04063
t=0.91  r=0.15593  dh=0.043092  max|ux|=0.04309  max|uy|=0.04309
t=1.01  r=0.15777  dh=0.044930  max|ux|=0.04493  max|uy|=0.04493
t=1.11  r=0.15897  dh=0.046132  max|ux|=0.04613  max|uy|=0.04613
t=1.21  r=0.15953  dh=0.046